# Lecture 8 — Matrix Product States

---

## Overview

In lecture 07 we saw that the entanglement entropy controls how efficiently a quantum state can be represented. In this lecture we implement that idea directly: we take a many-body wavefunction from exact diagonalisation and decompose it into a **Matrix Product State (MPS)** by performing a sequence of singular value decompositions.

This is the bridge between exact diagonalisation and DMRG. By the end you will understand what an MPS is, why it is efficient for low-entanglement states, and exactly what information is lost when we truncate the bond dimension.

---

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})


def build_tfim(N, J=1.0, Gamma=1.0):
    dim = 2**N
    rows, cols, data = [], [], []
    for state in range(dim):
        diag = 0.0
        for i in range(N - 1):
            sz_i = 1 if (state >> i) & 1 else -1
            sz_j = 1 if (state >> (i + 1)) & 1 else -1
            diag -= J * sz_i * sz_j
        if diag != 0.0:
            rows.append(state); cols.append(state); data.append(diag)
        for i in range(N):
            rows.append(state); cols.append(state ^ (1 << i)); data.append(-Gamma)
    return sp.csr_matrix((data, (rows, cols)), shape=(dim, dim))

## 1. The Problem with the Full Wavefunction

The ground state of a 20-site TFIM chain has $2^{20} \approx 10^6$ complex coefficients. But for a gapped system, entanglement is local — most of those coefficients are nearly zero. The MPS representation makes this structure explicit.

---

## 2. What Is a Matrix Product State?

An MPS writes the many-body wavefunction as a product of matrices:

$$|\psi\rangle = \sum_{s_1, \ldots, s_N} A^{s_1}_1 A^{s_2}_2 \cdots A^{s_N}_N\, |s_1 s_2 \cdots s_N\rangle$$

Each $A^{s_i}_i$ is a $\chi_{i-1} \times \chi_i$ matrix ($\chi_0 = \chi_N = 1$ at the boundaries). The **bond dimension** $\chi$ controls expressiveness: an MPS has $O(N\chi^2 d)$ parameters — polynomial in $N$ and $\chi$.

---

## 3. Every State Is an MPS (Exactly)

Every quantum state can be written exactly as an MPS with bond dimension up to $d^{N/2}$. The approximation arises when we **truncate**: keep only the $\chi$ largest singular values at each bond. If the discarded values are small (gapped states), the error is negligible.

---

## 4. Constructing an MPS by Sequential SVD

Given the coefficient tensor $\psi_{s_1 \cdots s_N}$:

1. Reshape to $d \times d^{N-1}$, SVD → $A^{s_1}_1$ and Schmidt coefficients at cut 1.
2. Form $\Sigma V^\dagger$, reshape to $(\chi_1 d) \times d^{N-2}$, SVD again.
3. Repeat until all sites processed.

Truncate each SVD to the $\chi$ largest singular values for an approximate MPS.

---

In [ ]:
def wavefunction_to_mps(psi, N, chi_max=None):
    """
    Convert a full wavefunction to MPS form via sequential SVD.
    Returns list of tensors A[i] with shape (chi_left, d, chi_right)
    and the truncation errors at each bond.
    """
    d = 2  # local Hilbert space dimension
    tensors = []
    trunc_errors = []

    # Start with the full wavefunction as a vector
    C = psi.copy().reshape(1, d**N)  # (1, d^N)

    chi_left = 1
    for i in range(N - 1):
        # Reshape: (chi_left * d) x d^(N-1-i)
        C = C.reshape(chi_left * d, d**(N - 1 - i))

        # SVD
        U, s, Vt = np.linalg.svd(C, full_matrices=False)

        # Truncate
        chi = len(s) if chi_max is None else min(len(s), chi_max)
        trunc_err = np.sum(s[chi:]**2) if chi < len(s) else 0.0
        trunc_errors.append(trunc_err)

        U = U[:, :chi]
        s = s[:chi]
        Vt = Vt[:chi, :]

        # Reshape U to (chi_left, d, chi) and store
        tensors.append(U.reshape(chi_left, d, chi))
        chi_left = chi

        # Pass sigma * Vt to next step
        C = np.diag(s) @ Vt

    # Last site: C has shape (chi_left, d)
    tensors.append(C.reshape(chi_left, d, 1))
    trunc_errors.append(0.0)

    return tensors, trunc_errors


def mps_overlap(tensors_a, tensors_b):
    """Compute <a|b> by contracting MPS tensors left to right."""
    N = len(tensors_a)
    # Left boundary: scalar 1
    L = np.array([[1.0]])
    for i in range(N):
        # tensors shape: (chi_l, d, chi_r)
        A = tensors_a[i]  # (chi_l_a, d, chi_r_a)
        B = tensors_b[i]  # (chi_l_b, d, chi_r_b)
        # Contract: L(a,b) * A*(a,s,a') * B(b,s,b') -> L'(a',b')
        tmp = np.einsum('ab,asc->bsc', L, A.conj())  # (chi_l_b, d, chi_r_a)
        L = np.einsum('bsc,bsd->cd', tmp, B)          # (chi_r_a, chi_r_b)
    return float(L[0, 0].real)


# Test: exact MPS reconstruction
N = 10
H = build_tfim(N, J=1.0, Gamma=1.0)
_, evecs = eigsh(H, k=1, which='SA')
psi0 = evecs[:, 0]

tensors_exact, _ = wavefunction_to_mps(psi0, N, chi_max=None)
overlap = mps_overlap(tensors_exact, tensors_exact)
max_bond = max(t.shape[2] for t in tensors_exact[:-1])

print(f"N={N}, exact MPS: max bond dimension = {max_bond}")
print(f"<MPS|MPS> = {overlap:.8f} (should be 1.0)")

## 5. The Truncation Error

At each cut $i$, the truncation error is $\epsilon_i = \sum_{\alpha > \chi} \lambda_\alpha^2$ — the squared norm of the discarded component. The entanglement entropy directly tells us how quickly this decays with $\chi$.

---

In [ ]:
# Truncation error vs chi for three phases
N = 14
chi_values = [1, 2, 4, 8, 16, 32]

fig, ax = plt.subplots(figsize=(8, 5))

for g, label, color in [(0.3, 'Ordered', 'tab:blue'), (1.0, 'Critical', 'tab:red'), (3.0, 'Disordered', 'tab:green')]:
    H = build_tfim(N, J=1.0, Gamma=g)
    _, evecs = eigsh(H, k=1, which='SA')
    psi0 = evecs[:, 0]

    total_errors = []
    for chi in chi_values:
        _, errs = wavefunction_to_mps(psi0, N, chi_max=chi)
        total_errors.append(sum(errs))

    ax.semilogy(chi_values, total_errors, 'o-', color=color, label=label)

ax.set_xlabel(r'Bond dimension $\chi$')
ax.set_ylabel('Total truncation error')
ax.set_title(f'Truncation error vs bond dimension, N={N}')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Canonical Form

An MPS is in **left-canonical form** when $\sum_{s} (A^s_i)^\dagger A^s_i = I$. The sequential SVD construction naturally produces a left-canonical MPS. In canonical form, expectation values can be computed efficiently and the singular values at each cut are directly accessible.

---

## 7. Computing Expectation Values from MPS

Computing a local observable $\langle \hat{O}_i \rangle$ from an MPS costs $O(N\chi^2 d^2)$ — linear in $N$, polynomial in $\chi$. The entanglement entropy at any cut is $S_A = -\sum_\alpha \lambda_\alpha^2 \log \lambda_\alpha^2$ read directly from the singular values.

---

In [ ]:
def mps_entropy_profile(psi, N):
    """Compute entanglement entropy at each bond via sequential SVD (exact)."""
    d = 2
    entropies = []
    C = psi.copy().reshape(1, d**N)
    chi_left = 1
    for i in range(N - 1):
        C = C.reshape(chi_left * d, d**(N - 1 - i))
        _, s, Vt = np.linalg.svd(C, full_matrices=False)
        s2 = s**2
        s2 = s2[s2 > 1e-15]
        S = -np.sum(s2 * np.log(s2))
        entropies.append(S)
        chi_left = len(s)
        C = np.diag(s) @ Vt
    return entropies


N = 18
fig, ax = plt.subplots(figsize=(8, 5))

for g, label, color in [(0.3, 'Ordered', 'tab:blue'), (1.0, 'Critical', 'tab:red'), (3.0, 'Disordered', 'tab:green')]:
    H = build_tfim(N, J=1.0, Gamma=g)
    _, evecs = eigsh(H, k=1, which='SA')
    S_profile = mps_entropy_profile(evecs[:, 0], N)
    ax.plot(range(1, N), S_profile, 'o-', color=color, label=label, markersize=4)

ax.set_xlabel('Cut position $\\ell$')
ax.set_ylabel(r'$S_A(\ell)$')
ax.set_title(f'Entanglement entropy from MPS SVD, N={N}')
ax.legend()
plt.tight_layout()
plt.show()

## 8. What Survives Truncation

- **Gapped phases**: Schmidt spectrum decays rapidly — even $\chi = 8$ captures the ground state energy to high precision.
- **Critical point**: spectrum decays slowly (power law). $\chi$ must grow as $\ell^{c/6}$ to maintain fixed accuracy.

This is not a failure — it is physics. The sequential SVD here is not just pedagogical: it is exactly the step that runs inside DMRG every time a site tensor is updated. Rather than starting from a full ED wavefunction and compressing it, DMRG directly optimises an MPS ansatz to find the ground state without ever constructing the exponentially large full wavefunction — the subject of lecture 09.

---

In [ ]:
# Ground state energy error vs chi for the three phases
N = 16
chi_values = [1, 2, 4, 8, 16, 32, 64]

def mps_to_vector(tensors):
    """Contract MPS back to a full wavefunction vector."""
    vec = tensors[0][:, :, :]  # (1, d, chi)
    vec = vec.reshape(2, -1)    # (d, chi)
    for i in range(1, len(tensors)):
        A = tensors[i]  # (chi_l, d, chi_r)
        # vec: (d^i, chi_l), A: (chi_l, d, chi_r) -> (d^i * d, chi_r)
        vec = np.einsum('ac,cdr->adr', vec, A).reshape(-1, A.shape[2])
    return vec.flatten()

fig, ax = plt.subplots(figsize=(8, 5))

for g, label, color in [(0.3, 'Ordered (Γ/J=0.3)', 'tab:blue'), (1.0, 'Critical (Γ/J=1.0)', 'tab:red'), (3.0, 'Disordered (Γ/J=3.0)', 'tab:green')]:
    H = build_tfim(N, J=1.0, Gamma=g)
    evals_exact, evecs_exact = eigsh(H, k=1, which='SA')
    psi0 = evecs_exact[:, 0]
    E0_exact = evals_exact[0]

    errors = []
    for chi in chi_values:
        tensors, _ = wavefunction_to_mps(psi0, N, chi_max=chi)
        psi_approx = mps_to_vector(tensors)
        # Normalise
        psi_approx /= np.linalg.norm(psi_approx)
        E_approx = float(psi_approx @ H @ psi_approx)
        errors.append(abs(E_approx - E0_exact) / abs(E0_exact))

    ax.semilogy(chi_values, errors, 'o-', color=color, label=label)

ax.set_xlabel(r'Bond dimension $\chi$')
ax.set_ylabel('Relative energy error')
ax.set_title(f'Energy error vs $\\chi$, N={N}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()